# E2: Thermal-only YOLO baseline

This notebook uses only IR/thermal images and the frozen recording-level split. It does not implement RGB-thermal fusion. The default stage is `validate`; training and held-out test evaluation require an explicit stage change.

In [1]:
from __future__ import annotations

import csv
import hashlib
import io
import shutil
import json
import math
import os
import random
import tempfile
import zipfile
from collections import Counter
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    drive = None

if drive is not None:
    drive.mount("/content/drive", force_remount=False)

PROJECT_ROOT = Path("/content/UAV_SAR")
if not PROJECT_ROOT.is_dir():
    PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks" and (PROJECT_ROOT.parent / "results").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
SPLIT_MANIFEST = PROJECT_ROOT / "results/dataset_split/recording_split_manifest.csv"
OUTPUT_ROOT = PROJECT_ROOT / "results/thermal_baseline"
PAIR_MANIFEST = OUTPUT_ROOT / "thermal_dataset_manifest.csv"
ZIP_CANDIDATES = [
    Path("/content/drive/MyDrive/WiSARDv1.zip"),
    Path("/content/drive/MyDrive/WiSARD/WiSARDv1.zip"),
]
ZIP_PATH = next((path for path in ZIP_CANDIDATES if path.is_file()), ZIP_CANDIDATES[0])
DRIVE_ROOT = Path("/content/drive/MyDrive/UAV_SAR_E2")

# Safe default: preparation/validation only. Change explicitly to run training.
STAGE = "validate"  # validate | prepare_manifest | materialize_development | train_cv | train_final | evaluate_test | smoke_tests
SELECTED_FOLD = "210417_MtErie_Enterprise"
SELECTED_CHUNK = 1
ALLOW_TEST_REEVALUATION = False
TOTAL_EPOCHS = 100
CHUNK_EPOCHS = 10
IMAGE_SIZE = 640
BATCH_SIZE = 8
SEED = 20260905
TEST_CONTEXTS = {"210327_Airfield_FLIR", "210812_Hannegan_Enterprise"}
DEVELOPMENT_CONTEXTS = {
    "210417_MtErie_Enterprise",
    "210529_Carnation_Enterprise",
    "210924_FHL_Enterprise",
    "220109_Baker_Enterprise",
}
PAIR_COLUMNS = [
    "image_member_path", "annotation_member_path", "recording_name",
    "recording_path", "collection_context", "partition", "modality",
    "image_mode", "image_width", "image_height", "image_extrema",
]
THERMAL_FRAME_OFFSET = 1  # audited filename rule only; never changes box coordinates
print(f"Project root: {PROJECT_ROOT}")
print(f"E2 stage: {STAGE}")


Project root: d:\Hackathons\miniproject5sem\UAV-Search-and-Rescue\notebooks
E2 stage: validate


In [2]:
def atomic_write(path: Path, data: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, temporary = tempfile.mkstemp(prefix=f".{path.name}.", dir=path.parent)
    try:
        with os.fdopen(fd, "w", encoding="utf-8", newline="") as handle:
            handle.write(data)
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(temporary, path)
    finally:
        if os.path.exists(temporary):
            os.unlink(temporary)


def manifest_fingerprint(rows: list[dict[str, str]]) -> str:
    payload = "\n".join(
        "|".join(row[column] for column in PAIR_COLUMNS) for row in rows
    )
    return hashlib.sha256(payload.encode()).hexdigest()


def read_frozen_split() -> list[dict[str, str]]:
    if not SPLIT_MANIFEST.is_file():
        raise FileNotFoundError(f"Frozen split is missing: {SPLIT_MANIFEST}")
    with SPLIT_MANIFEST.open(newline="", encoding="utf-8") as handle:
        rows = list(csv.DictReader(handle))
    required = {"recording_name", "recording_path", "collection_context", "modality", "partition"}
    if not rows or not required.issubset(rows[0]):
        raise ValueError(f"Frozen split must contain {sorted(required)}")
    thermal = [row for row in rows if row["modality"] == "IR"]
    if not thermal:
        raise ValueError("Frozen split contains no IR recordings.")
    if any(row["partition"] not in {"development", "test"} for row in thermal):
        raise ValueError("Frozen split contains an unsupported partition.")
    return sorted(thermal, key=lambda row: (row["recording_path"], row["recording_name"]))


def validate_annotation(archive: zipfile.ZipFile, member: str) -> tuple[list[int], list[str]]:
    classes, issues = [], []
    try:
        text = archive.read(member).decode("utf-8", errors="replace")
    except (KeyError, OSError, RuntimeError) as error:
        return [], [f"{member}: unreadable annotation: {error}"]
    for line_number, raw in enumerate(text.splitlines(), 1):
        fields = raw.split()
        if not fields:
            continue
        try:
            if len(fields) != 5:
                raise ValueError("expected five YOLO fields")
            class_value = float(fields[0])
            if not math.isfinite(class_value) or not class_value.is_integer() or int(class_value) != 0:
                raise ValueError("class_id must be the integer 0")
            values = [float(value) for value in fields[1:]]
            if not all(math.isfinite(value) for value in values):
                raise ValueError("coordinates must be finite")
            x_center, y_center, width, height = values
            if not 0 <= x_center <= 1 or not 0 <= y_center <= 1:
                raise ValueError("center coordinates must be in [0, 1]")
            if not 0 < width <= 1 or not 0 < height <= 1:
                raise ValueError("width and height must be in (0, 1]")
            if x_center - width / 2 < 0 or x_center + width / 2 > 1 or y_center - height / 2 < 0 or y_center + height / 2 > 1:
                raise ValueError("box edges must remain within normalized image bounds")
            classes.append(0)
        except (TypeError, ValueError, OverflowError) as error:
            issues.append(f"{member}:{line_number}: {error}")
    return classes, issues


def safe_target(root: Path, member: str) -> Path:
    normalized = Path(member.replace("\\", "/"))
    if normalized.is_absolute() or ".." in normalized.parts:
        raise ValueError(f"Unsafe archive member: {member}")
    root_resolved = root.resolve()
    target = (root_resolved / normalized).resolve()
    if target == root_resolved or root_resolved not in target.parents:
        raise ValueError(f"Archive member escapes target: {member}")
    return target


def materialized_image_path(root: Path, row: dict[str, str]) -> Path:
    return safe_target(root / "images" / row["partition"], row["image_member_path"])


def validate_image_list(paths: list[Path]) -> None:
    missing = [str(path) for path in paths if not path.is_file()]
    if missing:
        raise RuntimeError(f"Generated image list contains missing materialized files: {missing[:3]}")


def inspect_image_member(archive: zipfile.ZipFile, member: str) -> dict[str, object]:
    try:
        from PIL import Image
    except ImportError as error:
        raise RuntimeError("Pillow is required for thermal image validation; install the project image stack before preparation.") from error
    payload = archive.read(member)
    try:
        with Image.open(io.BytesIO(payload)) as image:
            image.verify()
        with Image.open(io.BytesIO(payload)) as image:
            extrema = image.getextrema()
            return {"mode": image.mode, "width": image.width, "height": image.height, "extrema": extrema}
    except Exception as error:
        raise ValueError(f"Unreadable thermal image {member}: {error}") from error


def prepare_thermal_manifest() -> list[dict[str, str]]:
    frozen_rows = read_frozen_split()
    if not ZIP_PATH.is_file():
        raise FileNotFoundError(f"Thermal ZIP is missing: {ZIP_PATH}")
    manifest, exclusions, classes = [], [], Counter()
    with zipfile.ZipFile(ZIP_PATH) as archive:
        names = [info.filename.replace("\\", "/") for info in archive.infolist()]
        by_recording: dict[str, list[str]] = {}
        for name in names:
            if Path(name).parts:
                by_recording.setdefault(Path(name).parts[0], []).append(name)
        for frozen in frozen_rows:
            members = by_recording.get(frozen["recording_name"], [])
            image_names = [name for name in members if Path(name).suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}]
            label_names = [name for name in members if Path(name).suffix.lower() in {".txt", ".ann", ".label", ".labels"} and Path(name).name.lower() != "count.txt"]
            images = {}
            labels = {}
            for name in image_names:
                stem = Path(name).stem
                if stem in images:
                    raise ValueError(f"Duplicate thermal image stem: {name}")
                images[stem] = name
            for name in label_names:
                stem = Path(name).stem
                if stem in labels:
                    raise ValueError(f"Duplicate thermal annotation stem: {name}")
                labels[stem] = name
            for stem, image in sorted(images.items()):
                label = labels.get(stem)
                if label is None:
                    exclusions.append({"recording_name": frozen["recording_name"], "image_member_path": image, "reason": "missing annotation"})
                    continue
                try:
                    image_info = inspect_image_member(archive, image)
                except (KeyError, OSError, RuntimeError, ValueError) as error:
                    exclusions.append({"recording_name": frozen["recording_name"], "image_member_path": image, "reason": str(error)})
                    continue
                ids, issues = validate_annotation(archive, label)
                if issues:
                    exclusions.extend({"recording_name": frozen["recording_name"], "image_member_path": image, "reason": issue} for issue in issues)
                    continue
                classes.update(ids)
                manifest.append({
                    "image_member_path": image,
                    "annotation_member_path": label,
                    "recording_name": frozen["recording_name"],
                    "recording_path": frozen["recording_path"],
                    "collection_context": frozen["collection_context"],
                    "partition": frozen["partition"],
                    "modality": "IR",
                    "image_mode": str(image_info["mode"]),
                    "image_width": str(image_info["width"]),
                    "image_height": str(image_info["height"]),
                    "image_extrema": repr(image_info["extrema"]),
                })
    manifest.sort(key=lambda row: (row["partition"], row["recording_path"], row["image_member_path"]))
    if not manifest or not any(row["partition"] == "development" for row in manifest) or not any(row["partition"] == "test" for row in manifest):
        raise ValueError("Thermal preparation did not produce usable development and test samples.")
    csv_buffer = io.StringIO()
    writer = csv.DictWriter(csv_buffer, fieldnames=PAIR_COLUMNS, lineterminator="\n")
    writer.writeheader()
    writer.writerows(manifest)
    if PAIR_MANIFEST.is_file():
        existing = PAIR_MANIFEST.read_bytes()
        if existing != csv_buffer.getvalue().encode():
            raise FileExistsError(f"Refusing to replace an existing incompatible thermal manifest: {PAIR_MANIFEST}")
    atomic_write(PAIR_MANIFEST, csv_buffer.getvalue())
    atomic_write(OUTPUT_ROOT / "thermal_exclusions.json", json.dumps(exclusions, indent=2))
    atomic_write(OUTPUT_ROOT / "thermal_manifest_metadata.json", json.dumps({
        "fingerprint": manifest_fingerprint(manifest),
        "pair_count": len(manifest),
        "class_ids": dict(classes),
        "zip_path": str(ZIP_PATH),
    }, indent=2))
    print(f"Prepared {len(manifest):,} thermal pairs; exclusions: {len(exclusions):,}; classes: {dict(classes)}")
    return manifest


def load_existing_manifest() -> list[dict[str, str]]:
    if not PAIR_MANIFEST.is_file():
        raise FileNotFoundError(f"Thermal manifest is missing: {PAIR_MANIFEST}; run STAGE='prepare_manifest'.")
    with PAIR_MANIFEST.open(newline="", encoding="utf-8") as handle:
        rows = list(csv.DictReader(handle))
    if not rows or set(PAIR_COLUMNS) - set(rows[0]):
        raise ValueError(f"Existing thermal manifest has an invalid schema: {PAIR_MANIFEST}")
    frozen_by_recording = {row["recording_name"]: row for row in read_frozen_split()}
    for row in rows:
        frozen = frozen_by_recording.get(row["recording_name"])
        if frozen is None or row["modality"] != "IR" or any(row[field] != frozen[field] for field in ("recording_path", "collection_context", "partition")):
            raise ValueError(f"Thermal manifest does not match frozen split: {row}")
    return rows


def validate_existing_materialization(rows: list[dict[str, str]], root: Path) -> None:
    integrity_path = root / "materialization_integrity.json"
    if not integrity_path.is_file():
        print(f"No materialization integrity record at {integrity_path}")
        return
    integrity = json.loads(integrity_path.read_text(encoding="utf-8"))
    missing = [path for path in integrity if not Path(path).is_file()]
    if missing:
        raise RuntimeError(f"Materialization integrity record references missing files: {missing[:3]}")
    print(f"Materialization integrity: PASS ({len(integrity):,} records)")


manifest: list[dict[str, str]] = []


Thermal ZIP not found at \content\drive\MyDrive\WiSARDv1.zip; manifest preparation is pending and no data/training action was taken.


In [3]:
def materialize(rows: list[dict[str, str]], root: Path) -> None:
    root.mkdir(parents=True, exist_ok=True)
    partition_paths: dict[str, list[Path]] = {"development": [], "test": []}
    integrity_path = root / "materialization_integrity.json"
    integrity: dict[str, dict[str, str]] = json.loads(integrity_path.read_text(encoding="utf-8")) if integrity_path.is_file() else {}
    with zipfile.ZipFile(ZIP_PATH) as archive:
        for row in rows:
            image_target = materialized_image_path(root, row)
            label_target = safe_target(root / "labels" / row["partition"], row["annotation_member_path"])
            image_target.parent.mkdir(parents=True, exist_ok=True)
            label_target.parent.mkdir(parents=True, exist_ok=True)
            image_bytes = archive.read(row["image_member_path"])
            label_bytes = archive.read(row["annotation_member_path"])
            expected = {"image": hashlib.sha256(image_bytes).hexdigest(), "label": hashlib.sha256(label_bytes).hexdigest()}
            if image_target.exists() and label_target.exists():
                actual = {"image": hashlib.sha256(image_target.read_bytes()).hexdigest(), "label": hashlib.sha256(label_target.read_bytes()).hexdigest()}
                if actual != expected:
                    raise RuntimeError(f"Existing materialization differs from source archive: {image_target}")
            else:
                if image_target.exists() or label_target.exists():
                    raise RuntimeError(f"Partial materialization exists for {row['image_member_path']}")
                for target, payload in ((image_target, image_bytes), (label_target, label_bytes)):
                    temporary = target.with_name(f".{target.name}.tmp")
                    temporary.write_bytes(payload)
                    os.replace(temporary, target)
            integrity[str(image_target)] = expected
            partition_paths[row["partition"]].append(image_target)
    for partition, paths in partition_paths.items():
        if not paths:
            continue
        validate_image_list(paths)
        atomic_write(root / f"{partition}.txt", "\n".join(str(path) for path in paths) + "\n")
    atomic_write(integrity_path, json.dumps(integrity, indent=2, sort_keys=True))


def write_fold_data(rows: list[dict[str, str]], root: Path, validation_context: str) -> None:
    fold_root = root / "fold_data" / validation_context
    fold_root.mkdir(parents=True, exist_ok=True)
    development = [row for row in rows if row["partition"] == "development"]
    if validation_context == "all-development":
        train_rows, validation_rows = development, []
    else:
        train_rows = [row for row in development if row["collection_context"] != validation_context]
        validation_rows = [row for row in development if row["collection_context"] == validation_context]
        train_contexts = {row["collection_context"] for row in train_rows}
        validation_contexts = {row["collection_context"] for row in validation_rows}
        if train_contexts & validation_contexts or validation_context not in DEVELOPMENT_CONTEXTS:
            raise ValueError(f"Invalid grouped fold: train={train_contexts}, validation={validation_contexts}")
        if any(row["collection_context"] in TEST_CONTEXTS for row in train_rows + validation_rows):
            raise ValueError("Held-out test context entered a development fold")
    if not train_rows or (validation_context != "all-development" and not validation_rows):
        raise ValueError(f"Fold {validation_context} lacks train or validation samples")
    train_paths = [materialized_image_path(root, row) for row in train_rows]
    validation_paths = [materialized_image_path(root, row) for row in validation_rows]
    validate_image_list(train_paths + validation_paths)
    atomic_write(fold_root / "train.txt", "\n".join(str(path) for path in train_paths) + "\n")
    atomic_write(fold_root / "val.txt", "\n".join(str(path) for path in validation_paths) + "\n")
    atomic_write(fold_root / "data.yaml", f"path: {root}\ntrain: {fold_root / 'train.txt'}\nval: {fold_root / 'val.txt'}\nnc: 1\nnames: [person]\n")


def checkpoint_metadata(fold: str) -> dict[str, object]:
    split_fingerprint = hashlib.sha256(SPLIT_MANIFEST.read_bytes()).hexdigest() if SPLIT_MANIFEST.is_file() else "unavailable"
    return {
        "experiment": "E2_thermal_only",
        "fold": fold,
        "manifest_fingerprint": manifest_fingerprint(manifest),
        "split_fingerprint": split_fingerprint,
        "seed": SEED,
        "image_size": IMAGE_SIZE,
        "batch_size": BATCH_SIZE,
        "total_epochs": TOTAL_EPOCHS,
    }


def load_checkpoint_payload(path: Path) -> dict | None:
    try:
        import pickle
        import torch
        try:
            payload = torch.load(path, map_location="cpu", weights_only=False)
        except TypeError:
            payload = torch.load(path, map_location="cpu")
        if not isinstance(payload, dict):
            return None
        saved_epoch = payload.get("epoch")
        if not isinstance(saved_epoch, int) or saved_epoch < -1:
            return None
        return payload
    except (EOFError, OSError, RuntimeError, ValueError, pickle.UnpicklingError, AttributeError, ImportError, ModuleNotFoundError):
        return None


def readable_checkpoint(path: Path) -> bool:
    return load_checkpoint_payload(path) is not None


def checkpoint_completed_epochs(path: Path) -> int | None:
    payload = load_checkpoint_payload(path)
    if payload is None:
        return None
    return int(payload["epoch"]) + 1


def checkpoint_has_optimizer(path: Path) -> bool:
    payload = load_checkpoint_payload(path)
    return payload is not None and payload.get("optimizer") is not None and payload.get("train_args") is not None


def find_valid_checkpoint(run_root: Path, fold: str, validation_context: str | None = None) -> tuple[Path | None, int]:
    metadata_path = run_root / "checkpoint_metadata.json"
    checkpoint_candidates = (run_root / "weights/last.pt", run_root / "weights/last.previous.pt")
    if not metadata_path.is_file():
        if any(candidate.is_file() for candidate in checkpoint_candidates) or (run_root.exists() and any(run_root.iterdir())):
            raise RuntimeError(f"Existing run state lacks compatibility metadata in {run_root}; refusing a fresh restart.")
        return None, 0
    saved = json.loads(metadata_path.read_text(encoding="utf-8"))
    expected = checkpoint_metadata(fold)
    if validation_context is not None:
        expected["validation_context"] = validation_context
    if any(saved.get(key) != value for key, value in expected.items()):
        raise RuntimeError(f"Incompatible checkpoint metadata in {run_root}; refusing a fresh restart.")
    candidates = ((run_root / "weights/last.pt", "completed_epochs"), (run_root / "weights/last.previous.pt", "previous_completed_epochs"))
    metadata_changed = False
    for candidate, epoch_key in candidates:
        if candidate.is_file() and candidate.stat().st_size > 0:
            actual_completed = checkpoint_completed_epochs(candidate)
            if actual_completed is not None:
                if actual_completed == 0 and isinstance(saved.get(epoch_key), int) and saved[epoch_key] > 0:
                    actual_completed = saved[epoch_key]
                if saved.get(epoch_key) != actual_completed:
                    saved[epoch_key] = actual_completed
                    metadata_changed = True
                if metadata_changed:
                    atomic_write(metadata_path, json.dumps(saved, indent=2))
                return candidate, actual_completed
            print(f"Ignoring unreadable or epoch-incomplete checkpoint and trying fallback: {candidate}")
    raise RuntimeError(f"Checkpoint metadata exists but no valid checkpoint is available in {run_root}; refusing pretrained restart.")


def chunk_target(completed: int) -> int:
    return min(completed + CHUNK_EPOCHS, TOTAL_EPOCHS)


def train_selected_fold(fold: str, validation_context: str) -> None:
    if STAGE not in {"train_cv", "train_final"}:
        raise RuntimeError("Training is gated. Set STAGE explicitly to train_cv or train_final.")
    try:
        from ultralytics import YOLO
        import torch
    except ImportError as error:
        raise ImportError("Install the project-pinned Ultralytics environment before training.") from error
    chunk_number = SELECTED_CHUNK
    run_root = DRIVE_ROOT / "runs" / ("final" if fold == "final" else "cv") / validation_context / f"chunk_{chunk_number:02d}"
    control_root = DRIVE_ROOT / "control" / ("final" if fold == "final" else "cv") / validation_context
    control_root.mkdir(parents=True, exist_ok=True)
    checkpoint, completed = find_valid_checkpoint(run_root, fold, validation_context)
    if checkpoint is None and SELECTED_CHUNK > 1:
        previous_root = run_root.parent / f"chunk_{SELECTED_CHUNK - 1:02d}"
        previous_checkpoint = previous_root / "weights/best.pt"
        if not previous_checkpoint.is_file():
            previous_checkpoint = previous_root / "weights/last.pt"
        if not previous_checkpoint.is_file() or not readable_checkpoint(previous_checkpoint):
            raise RuntimeError(f"Previous E2 chunk is missing or invalid; refusing pretrained restart: {previous_root}")
        checkpoint, completed = previous_checkpoint, (SELECTED_CHUNK - 1) * CHUNK_EPOCHS
    if completed >= TOTAL_EPOCHS:
        print(f"{fold} already complete: {completed}/{TOTAL_EPOCHS}")
        return
    random.seed(SEED)
    torch.manual_seed(SEED)
    previous = run_root / "weights/last.previous.pt"
    latest = run_root / "weights/last.pt"
    if checkpoint == latest and latest.is_file():
        shutil.copy2(latest, previous)
        metadata = checkpoint_metadata(fold)
        metadata.update({"completed_epochs": completed, "previous_completed_epochs": completed, "validation_context": validation_context})
        atomic_write(control_root / f"chunk_{chunk_number:02d}.json", json.dumps(metadata, indent=2))
    resume_training = bool(checkpoint and checkpoint_has_optimizer(checkpoint))
    if checkpoint and not resume_training:
        print(f"Starting a new chunk from loadable weights: {checkpoint}; optimizer resume is unavailable.")
    initial_metadata = checkpoint_metadata(fold)
    initial_metadata.update({"validation_context": validation_context, "completed_epochs": completed, "previous_completed_epochs": completed, "status": "running"})
    atomic_write(control_root / f"chunk_{chunk_number:02d}.json", json.dumps(initial_metadata, indent=2))
    model = YOLO(str(checkpoint)) if checkpoint else YOLO("yolov8s.pt")
    write_fold_data(manifest, DRIVE_ROOT / "data", validation_context)
    target = chunk_target(completed)
    model.train(data=str(DRIVE_ROOT / "data" / "fold_data" / validation_context / "data.yaml"), project=str(run_root.parent), name=run_root.name, epochs=target if resume_training else CHUNK_EPOCHS, imgsz=IMAGE_SIZE, batch=BATCH_SIZE, seed=SEED, device=0 if torch.cuda.is_available() else "cpu", workers=2, cache=False, patience=0, deterministic=True, amp=True, close_mosaic=10 if target >= TOTAL_EPOCHS else 0, val=fold != "final", resume=resume_training, exist_ok=False)
    latest = run_root / "weights/last.pt"
    actual_completed = checkpoint_completed_epochs(latest)
    if actual_completed is None:
        raise RuntimeError(f"Training returned without a valid epoch-bearing checkpoint: {latest}")
    metadata = checkpoint_metadata(fold)
    metadata.update({"completed_epochs": actual_completed, "previous_completed_epochs": completed, "validation_context": validation_context, "partition": "development", "independent_validation": fold != "final", "status": "complete" if actual_completed >= TOTAL_EPOCHS else "partial", "checkpoint": str(latest), "optimizer_resume": resume_training})
    if not run_root.is_dir():
        raise RuntimeError(f"Ultralytics did not create the expected E2 chunk directory: {run_root}")
    atomic_write(run_root / "checkpoint_metadata.json", json.dumps(metadata, indent=2))
    print(f"{fold}: completed {actual_completed}/{TOTAL_EPOCHS}; checkpoint={latest}")


def evaluate_test_guard() -> None:
    if STAGE != "evaluate_test":
        raise RuntimeError("Held-out evaluation is gated; set STAGE='evaluate_test' explicitly.")
    result = OUTPUT_ROOT / "test_evaluation.json"
    if result.is_file() and not ALLOW_TEST_REEVALUATION:
        raise RuntimeError(f"Test evaluation already exists at {result}; set ALLOW_TEST_REEVALUATION=True to re-evaluate explicitly.")
    final_metadata = DRIVE_ROOT / "runs" / "final" / "all-development" / "chunk_10" / "checkpoint_metadata.json"
    if not final_metadata.is_file():
        raise RuntimeError("Final development model metadata is missing; test evaluation is not available.")
    metadata = json.loads(final_metadata.read_text(encoding="utf-8"))
    if metadata.get("experiment") != "E2_thermal_only" or metadata.get("status") != "complete" or metadata.get("partition") != "development" or metadata.get("independent_validation") is not False:
        raise RuntimeError("Final development model is incomplete or not development-only.")
    checkpoint = Path(metadata.get("checkpoint", ""))
    if not checkpoint.is_file() or not readable_checkpoint(checkpoint):
        raise RuntimeError(f"Final E2 checkpoint is missing or unreadable: {checkpoint}")
    if metadata.get("manifest_fingerprint") != manifest_fingerprint(manifest):
        raise RuntimeError("Final E2 checkpoint manifest fingerprint does not match the current thermal manifest.")
    if metadata.get("split_fingerprint") != hashlib.sha256(SPLIT_MANIFEST.read_bytes()).hexdigest():
        raise RuntimeError("Final E2 checkpoint split fingerprint does not match the frozen split.")
    return metadata


def evaluate_test() -> None:
    metadata = evaluate_test_guard()
    try:
        from ultralytics import YOLO
    except ImportError as error:
        raise ImportError("Install the project-pinned Ultralytics environment before test evaluation.") from error
    materialize(manifest, DRIVE_ROOT / "data")
    test_rows = [row for row in manifest if row["partition"] == "test"]
    if not test_rows:
        raise RuntimeError("Test evaluation requires eligible supervised thermal test rows.")
    test_root = DRIVE_ROOT / "data" / "test_evaluation"
    test_paths = [materialized_image_path(DRIVE_ROOT / "data", row) for row in test_rows]
    validate_image_list(test_paths)
    atomic_write(test_root / "test.txt", "\n".join(str(path) for path in test_paths) + "\n")
    atomic_write(test_root / "data.yaml", f"path: {DRIVE_ROOT / 'data'}\ntest: {test_root / 'test.txt'}\nnc: 1\nnames: [person]\n")
    model = YOLO(str(metadata["checkpoint"]))
    results = model.val(data=str(test_root / "data.yaml"), split="test", project=str(OUTPUT_ROOT), name="test_evaluation", imgsz=IMAGE_SIZE, batch=BATCH_SIZE, workers=2, plots=True)
    metrics = dict(getattr(results, "results_dict", {}))
    atomic_write(OUTPUT_ROOT / "test_evaluation.json", json.dumps({"metadata": metadata, "test_contexts": sorted(TEST_CONTEXTS), "metrics": metrics}, indent=2, default=float))
    print(f"Held-out test evaluation written to {OUTPUT_ROOT / 'test_evaluation.json'}")




Validation-only mode: no materialization, training, or test evaluation requested.


In [ ]:
from pathlib import Path
import tempfile
import zipfile


def run_e2_regression_smoke_tests() -> None:
    import pickle
    import sys
    import types

    try:
        import torch
        synthetic_torch = torch
    except ImportError:
        synthetic_torch = types.SimpleNamespace(
            save=lambda payload, path: pickle.dump(payload, Path(path).open("wb")),
            load=lambda path, **_: pickle.load(Path(path).open("rb")),
        )
        sys.modules["torch"] = synthetic_torch

    original_zip = globals()["ZIP_PATH"]
    def check_label(text: str, valid: bool) -> None:
        label_path = Path(tempfile.mkdtemp()) / "label.txt"
        archive_path = label_path.parent / "labels.zip"
        with zipfile.ZipFile(archive_path, "w") as archive:
            archive.writestr("label.txt", text)
        with zipfile.ZipFile(archive_path) as archive:
            _, issues = validate_annotation(archive, "label.txt")
        assert (not issues) is valid, (text, issues)

    check_label("0 0.5 0.5 0.25 0.25\n", True)
    check_label("", True)
    for invalid in ("1 0.5 0.5 0.25 0.25", "0.5 0.5 0.5 0.25 0.25", "0 0.5 0.5", "0 nan 0.5 0.2 0.2", "0 0.5 0.5 0 0.2", "0 0.1 0.1 0.4 0.4"):
        check_label(invalid, False)
    try:
        with tempfile.TemporaryDirectory() as temporary:
            root = Path(temporary)
            archive_path = root / "synthetic.zip"
            rows = [{"image_member_path": "recording/nested/frame_001.png", "annotation_member_path": "recording/nested/frame_001.txt", "recording_name": "recording", "recording_path": "recording", "collection_context": "210417_MtErie_Enterprise", "partition": "development", "modality": "IR", "image_mode": "L", "image_width": "1", "image_height": "1", "image_extrema": "(0, 0)"}]
            with zipfile.ZipFile(archive_path, "w") as archive:
                archive.writestr(rows[0]["image_member_path"], b"thermal")
                archive.writestr(rows[0]["annotation_member_path"], "0 0.5 0.5 0.5 0.5\n")
            globals()["ZIP_PATH"] = archive_path
            materialize(rows, root / "data")
            expected = materialized_image_path(root / "data", rows[0])
            assert expected.is_file()
            assert str(expected) in (root / "data/development.txt").read_text()
            write_fold_data(rows, root / "data", "all-development")
            assert str(expected) in (root / "data/fold_data/all-development/train.txt").read_text()

            run_root = root / "run"
            (run_root / "weights").mkdir(parents=True)
            latest = run_root / "weights/last.pt"
            previous = run_root / "weights/last.previous.pt"
            synthetic_torch.save({"epoch": 14, "model": {"synthetic": True}}, latest)
            synthetic_torch.save({"epoch": 9, "model": {"synthetic": True}}, previous)
            saved = checkpoint_metadata("smoke")
            saved.update({"completed_epochs": 10, "previous_completed_epochs": 10})
            metadata_path = run_root / "checkpoint_metadata.json"
            metadata_path.write_text(json.dumps(saved))

            selected, epoch = find_valid_checkpoint(run_root, "smoke")
            assert selected == latest and epoch == 15
            repaired = json.loads(metadata_path.read_text())
            assert repaired["completed_epochs"] == 15
            assert repaired["previous_completed_epochs"] == 10
            assert chunk_target(epoch) == 25

            synthetic_torch.save({"epoch": -1, "model": {"synthetic": True}}, latest)
            selected, epoch = find_valid_checkpoint(run_root, "smoke")
            assert selected == latest and epoch == 15
            assert not checkpoint_has_optimizer(latest)

            latest.write_bytes(b"corrupt")
            selected, epoch = find_valid_checkpoint(run_root, "smoke")
            assert selected == previous and epoch == 10
            assert chunk_target(epoch) == 20

            previous.write_bytes(b"also corrupt")
            try:
                find_valid_checkpoint(run_root, "smoke")
            except RuntimeError:
                pass
            else:
                raise AssertionError("No-valid-checkpoint case restarted silently")
    finally:
        globals()["ZIP_PATH"] = original_zip
        if synthetic_torch.__class__.__module__ == "types":
            sys.modules.pop("torch", None)


def run_selected_stage() -> None:
    global manifest
    if STAGE == "validate":
        frozen = read_frozen_split()
        print(f"Frozen split validation: PASS ({len(frozen)} IR rows)")
        if PAIR_MANIFEST.is_file():
            manifest = load_existing_manifest()
            validate_existing_materialization(manifest, DRIVE_ROOT / "data")
            print(f"Existing thermal manifest: PASS ({len(manifest):,} pairs)")
        else:
            print(f"Thermal manifest unavailable: run STAGE='prepare_manifest' explicitly: {PAIR_MANIFEST}")
        return
    if STAGE == "prepare_manifest":
        if not ZIP_PATH.is_file():
            raise FileNotFoundError(f"Thermal ZIP is missing: {ZIP_PATH}")
        manifest = prepare_thermal_manifest()
        return
    if STAGE in {"materialize_development", "train_cv", "train_final", "evaluate_test"}:
        manifest = load_existing_manifest()
    if STAGE == "materialize_development":
        development = [row for row in manifest if row["partition"] == "development"]
        materialize(development, DRIVE_ROOT / "data")
        print(f"Materialized {len(development):,} development pairs; test rows were not materialized.")
        return
    if STAGE == "train_cv":
        if SELECTED_FOLD not in DEVELOPMENT_CONTEXTS:
            raise ValueError(f"SELECTED_FOLD must be one development context, got {SELECTED_FOLD}")
        development = [row for row in manifest if row["partition"] == "development"]
        materialize(development, DRIVE_ROOT / "data")
        train_selected_fold(SELECTED_FOLD, SELECTED_FOLD)
        return
    if STAGE == "train_final":
        development = [row for row in manifest if row["partition"] == "development"]
        materialize(development, DRIVE_ROOT / "data")
        train_selected_fold("final", "all-development")
        return
    if STAGE == "evaluate_test":
        evaluate_test()
        return
    if STAGE == "smoke_tests":
        run_e2_regression_smoke_tests()
        return
    raise ValueError(f"Unsupported E2 stage: {STAGE}")


run_selected_stage()
